# 🚀 Pipeline Tiền Xử Lý Dữ Liệu Đa Nguồn Cho Ngôn Ngữ Ký Hiệu ASL
### (High-Throughput Multi-Dataset ASL Preprocessor with GPU Acceleration)

Notebook này cung cấp quy trình tiền xử lý dữ liệu chuẩn hóa, hiệu năng cao, tối ưu cho **Kaggle 2× NVIDIA T4 GPU**, **Colab GPU**, hoặc **Kaggle CPU (224 vCPUs)**.

---

### 📌 Các Đặc Tính Kỹ Thuật Chính:
1. **Tự động hóa GPU song song (Multi-GPU Worker Striping)**:
   - Tự động nhận diện GPU (`cuda:0`, `cuda:1`) và phân phối các luồng xử lý độc lập, tối ưu hóa VRAM và thông lượng.
2. **Tập dữ liệu áp dụng**:
   - `ASL Alphabet`: 87,000 ảnh tĩnh (lặp lại thành $T=40$ khung hình chuẩn).
   - `Synthetic ASL Numbers`: 10,000 ảnh chữ số ($T=40$).
   - `WLASL`: 11,980 video từ vựng độc lập ($T \ge 40$).
   - `ASL Citizen`: 83,399 video từ vựng độc lập ($T \ge 40$).
   - `ChicagoFSWild`: Chuỗi video đánh vần chữ cái ngón tay liên tục.
   - *Lưu ý*: **How2Sign** đã được cấu hình tự động bỏ qua (`--disable-how2sign`) vì video gốc đang được tải riêng.
3. **Chuẩn hóa topo 60 điểm (Canonical 60-Keypoint Topology)**:
   - 21 điểm bàn tay trái + 21 điểm bàn tay phải + 6 điểm thân/vai + 12 điểm cơ mặt.
   - 9 kênh động học vi phân (Kinematics: Tọa độ, Vận tốc, Gia tốc theo bộ lọc Savitzky-Golay).
   - 19 đặc trưng ngữ âm học (ASL Phonology: Góc hướng lòng bàn tay, độ cong ngón tay, khoảng cách tương quan hai tay và khuôn mặt).
4. **Cơ chế bộ nhớ chống tràn đĩa (RAM-First / Scratch Buffer)**:
   - Tự động tận dụng `/dev/shm` (RAM) để nén zip tốc độ cực đại, tự động kiểm tra dung lượng để tránh tràn RAM trên máy 30GB.
   - Ghi định kỳ các shard `.pt` trực tiếp vào file `/kaggle/working/asl_unified_shards.zip`.

## 🛠️ Bước 1: Cài Đặt Thư Viện Cần Thiết
Cài đặt `rtmlib`, `onnxruntime-gpu` để tăng tốc độ phát hiện điểm khớp bằng nhân CUDA trên GPU T4.

In [ ]:
# 1. Cài đặt các thư viện phụ thuộc (hỗ trợ GPU)
!pip install -q rtmlib onnxruntime-gpu opencv-python pandas torch torchvision

## 🔍 Bước 2: Kiểm Tra Cấu Hình Phần Cứng Hệ Thống
Xác nhận hệ thống nhận diện đúng 2 GPU T4 (hoặc CPU đa luồng) và kiểm tra dung lượng RAM.

In [ ]:
import os
import shutil
import torch

print("=" * 65)
print("  KIỂM TRA CẤU HÌNH HỆ THỐNG RUNTIME")
print("=" * 65)
if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    print(f"[GPU] CUDA Khả dụng: {num_gpus} GPU")
    for i in range(num_gpus):
        name = torch.cuda.get_device_name(i)
        mem = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f"       - GPU {i}: {name} ({mem:.1f} GB VRAM)")
else:
    print("[GPU] Không phát hiện CUDA GPU. Script sẽ tự động chạy chế độ CPU đa luồng.")

print(f"[CPU] Số lượng lõi CPU khả dụng: {os.cpu_count()}")

if os.path.exists("/dev/shm"):
    total, used, free = shutil.disk_usage("/dev/shm")
    print(f"[RAM] Bộ nhớ chia sẻ (/dev/shm): Tổng {total/(1024**3):.1f} GB, Còn trống {free/(1024**3):.1f} GB")
print("=" * 65)

## 📂 Bước 3: Xác Minh Các Bộ Dữ Liệu Đầu Vào
Đảm bảo các dataset đã được gắn (add dataset) vào session Kaggle Notebook.

In [ ]:
from pathlib import Path

dataset_checks = {
    "ASL Alphabet": ["/kaggle/input/datasets/grassknoted/asl-alphabet", "/kaggle/input/asl-alphabet"],
    "Synthetic ASL Numbers": ["/kaggle/input/datasets/lexset/synthetic-asl-numbers", "/kaggle/input/synthetic-asl-numbers"],
    "WLASL": ["/kaggle/input/datasets/risangbaskoro/wlasl-processed", "/kaggle/input/wlasl-processed"],
    "ASL Citizen": ["/kaggle/input/datasets/abd0kamel/asl-citizen", "/kaggle/input/asl-citizen"],
    "ChicagoFSWild": ["/kaggle/input/datasets/joebeachcapital/chicagofswild", "/kaggle/input/chicagofswild"],
}

print("=" * 65)
print("  KIỂM TRA CÁC ĐƯỜNG DẪN DỮ LIỆU ĐẦU VÀO")
print("=" * 65)
for name, paths in dataset_checks.items():
    matched = None
    for p in paths:
        if Path(p).exists():
            matched = p
            break
    if matched:
        print(f"  [OK] {name:22s} -> {matched}")
    else:
        print(f"  [!] {name:22s} -> CHƯA TÌM THẤY (Hãy gắn dataset vào notebook nếu cần xử lý)")
print("=" * 65)

## 📝 Bước 4: Tạo Tệp Mã Nguồn Tiền Xử Lý (`unified_multidataset_preprocessor.py`)
Cell này sử dụng lệnh `%%writefile` để tự động dán toàn bộ mã nguồn xử lý tối ưu hóa mới nhất vào thư mục `preprocessing/`.

In [ ]:
# Tạo thư mục lưu trữ mã nguồn
import os
os.makedirs('preprocessing', exist_ok=True)


In [ ]:
%%writefile preprocessing/unified_multidataset_preprocessor.py
#!/usr/bin/env python3
"""
================================================================================
  HIGH-THROUGHPUT PARALLEL MULTI-DATASET ASL PREPROCESSOR & ZIP STREAMING
================================================================================
Production-grade parallel preprocessor leveraging all 96 vCPUs to ingest:
  1. How2Sign Holistic (35,142 continuous sentence sequences)
  2. ASL Citizen (83,399 isolated sign videos + ASL-LEX annotations)
  3. WLASL Processed (11,980 isolated gloss videos)
  4. Synthetic ASL Numbers (10 digit classes)
  5. ASL Alphabet (29 classes, all 87,000 images)

User Invariants:
  - NO CAPPING: Ingest all available samples across all classes.
  - MINIMUM 40 FRAMES: Every video and media sequence has T >= 40.
    Static images (Alphabet, Numbers) are repeated to T=40 static frames.
    Videos with T < 40 are smoothly linearly interpolated to T = 40.
  - 60-Keypoint Canonical Topology (21 LH, 21 RH, 6 Pose, 12 Face Mesh).
  - Reference-Part Normalization (mid-shoulder anchor, inter-shoulder scale).
  - 9D Kinematics (5-point quadratic zero-phase Savitzky-Golay velocity & acceleration).
  - 19D ASL Phonology (Palm normals, bimanual sync, face anchoring, finger curl).
  - Zero Disk Exhaustion: In-memory compression directly into a single .zip archive
    (/kaggle/working/asl_unified_shards.zip) with immediate RAM garbage collection.
  - Saturated Multiprocessing: Utilizes 48-64 parallel worker processes on CPU.
================================================================================
"""

import os
import sys
import io
import time
import json
import math
import glob
import zipfile
import argparse
import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any, Iterator, Union
import multiprocessing as mp

# Configure single-thread per worker process to prevent thread thrashing
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd
import torch

try:
    import cv2
except ImportError:
    cv2 = None

try:
    import rtmlib
    from rtmlib import Wholebody
    _RTMLIB_AVAILABLE = True
except ImportError:
    _RTMLIB_AVAILABLE = False


# ==============================================================================
# 1. CANONICAL 60-KEYPOINT TOPOLOGY & KINEMATICS
# ==============================================================================

H2S_POSE_INDICES = [500, 501, 512, 513, 502, 503]
H2S_FACE_INDICES = [1, 4, 152, 0, 33, 263, 61, 291, 10, 109, 338, 9]

RTMW_POSE_INDICES = [5, 6, 11, 12, 7, 8]
RTMW_FACE_INDICES = [23, 31, 37, 39, 41, 46, 52, 55, 74, 77, 85, 88]


def convert_543_to_60(arr_543: np.ndarray) -> np.ndarray:
    """Converts MediaPipe Holistic (T, 543, 3) to canonical (T, 60, 3)."""
    T = arr_543.shape[0]
    out = np.zeros((T, 60, 3), dtype=np.float32)
    out[:, 0:21, :] = arr_543[:, 468:489, :3]
    out[:, 21:42, :] = arr_543[:, 522:543, :3]
    out[:, 42:48, :] = arr_543[:, H2S_POSE_INDICES, :3]
    out[:, 48:60, :] = arr_543[:, H2S_FACE_INDICES, :3]
    return out


def convert_133_to_60(arr_133: np.ndarray) -> np.ndarray:
    """Converts RTMW Wholebody (T, 133, 2 or 3) to canonical (T, 60, 3)."""
    T, K, C = arr_133.shape
    out = np.zeros((T, 60, 3), dtype=np.float32)
    in_c = min(C, 3)
    out[:, 0:21, :in_c] = arr_133[:, 91:112, :in_c]
    out[:, 21:42, :in_c] = arr_133[:, 112:133, :in_c]
    out[:, 42:48, :in_c] = arr_133[:, RTMW_POSE_INDICES, :in_c]
    out[:, 48:60, :in_c] = arr_133[:, RTMW_FACE_INDICES, :in_c]
    return out


def ensure_min_40_frames(pos: np.ndarray, min_len: int = 40) -> np.ndarray:
    """
    Ensures landmark trajectory pos [T, 60, 3] has at least min_len (40) frames.
    - Static hand (T == 1): repeats identically for exactly min_len frames.
    - Short sequence (1 < T < min_len): smooth linear interpolation along time axis to min_len frames.
    - T >= min_len: leaves untouched.
    """
    T, K, C = pos.shape
    if T == 1:
        return np.repeat(pos, min_len, axis=0)
    elif T < min_len:
        t_old = np.linspace(0.0, 1.0, T)
        t_new = np.linspace(0.0, 1.0, min_len)
        pos_flat = pos.reshape(T, K * C)
        pos_interp = np.zeros((min_len, K * C), dtype=pos.dtype)
        for i in range(K * C):
            pos_interp[:, i] = np.interp(t_new, t_old, pos_flat[:, i])
        return pos_interp.reshape(min_len, K, C)
    return pos


def compute_kinematics_and_phonology(pos: np.ndarray, is_static: bool = False, fps: float = 30.0) -> Tuple[np.ndarray, np.ndarray]:
    """
    Computes Polished V4 Reference-Part Normalization, 9D kinematics, and 19D ASL phonology:
    1. EMA Sternum Anchoring + 3D Torso-Box Diagonal Scaling.
    2. Delta_t-scaled Savitzky-Golay Velocity & Acceleration with human biomechanical bounds.
    3. Standardized 19D Phonology with Gaussian Face Contact Kernel and Palm-Normalized Curls.
    """
    pos = ensure_min_40_frames(pos, min_len=40)
    T, K, _ = pos.shape

    # Construct valid mask
    val_mask = np.any(np.abs(pos[:, :, :2]) > 1e-4, axis=-1)  # [T, K]

    # Level 1 & 2: Anatomical Reference-Part Normalization with EMA Sternum Filter
    pos_scaled = pos.copy()
    pos_max = np.nanmax(np.abs(pos[:, :, :2]))
    if pos_max > 5.0:
        pos_scaled = pos / pos_max

    l_sh = pos_scaled[:, 42, :3]
    r_sh = pos_scaled[:, 43, :3]
    sh_present = (np.linalg.norm(l_sh[:, :2], axis=-1) > 0.01) & (np.linalg.norm(r_sh[:, :2], axis=-1) > 0.01)

    if sh_present.any():
        valid_idx = np.where(sh_present)[0]
        mid_sh_valid = (l_sh[sh_present] + r_sh[sh_present]) * 0.5
        d_sh = l_sh[sh_present] - r_sh[sh_present]
        sh_dist = np.sqrt(np.sum(d_sh ** 2, axis=-1))
        scale_ref = float(np.median(sh_dist)) if len(sh_dist) > 0 else 1.0

        # Torso diagonal scaling
        mid_sh_full = np.empty((T, 3), dtype=np.float32)
        if len(valid_idx) == T:
            mid_sh_full[:] = mid_sh_valid
        else:
            for c in range(3):
                mid_sh_full[:, c] = np.interp(np.arange(T), valid_idx, mid_sh_valid[:, c])

        alpha = 0.90
        anchor_seq = np.empty((T, 3), dtype=np.float32)
        anchor_seq[0] = mid_sh_full[0]
        for t in range(1, T):
            anchor_seq[t] = alpha * anchor_seq[t - 1] + (1.0 - alpha) * mid_sh_full[t]

        scale_ref = max(1e-3, scale_ref)
        inv_scale = np.float32(1.0 / scale_ref)
        norm_pos = (pos_scaled - anchor_seq[:, None, :]) * inv_scale
    else:
        # Fallback to face or hand centroid
        anchor = np.mean(pos_scaled[:, :21, :3], axis=1, keepdims=True)
        scale_ref = 1.0
        norm_pos = pos_scaled - anchor

    # Clean out of bounds or collapsed hands
    h_both = norm_pos[:, :42, :3].reshape(T, 2, 21, 3)
    wrist = h_both[:, :, 0:1, :]
    diff = h_both - wrist
    dist_sq = np.sum(diff * diff, axis=-1)
    has_exploded = np.max(dist_sq, axis=2) > 0.3025
    for h_idx, (hs, he) in enumerate([(0, 21), (21, 42)]):
        bad = has_exploded[:, h_idx]
        if bad.any():
            norm_pos[bad, hs:he, :] = 0.0

    kinematics = np.zeros((T, K, 9), dtype=np.float32)
    kinematics[:, :, :3] = norm_pos

    # 2. Savitzky-Golay Velocity & Acceleration with true delta_t scaling
    if not is_static and T >= 5:
        delta_t = 1.0 / max(fps, 1.0)
        v_clip = 12.0
        a_clip = 45.0

        p0 = norm_pos[:-4]
        p1 = norm_pos[1:-3]
        p2 = norm_pos[2:-2]
        p3 = norm_pos[3:-1]
        p4 = norm_pos[4:]

        v_mid = (-2.0 * p0 - p1 + p3 + 2.0 * p4) / (10.0 * delta_t)
        kinematics[2:-2, :, 3:6] = np.clip(v_mid, -v_clip, v_clip)
        kinematics[1, :, 3:6] = np.clip((norm_pos[2] - norm_pos[0]) / (2.0 * delta_t), -v_clip, v_clip)
        kinematics[0, :, 3:6] = np.clip((norm_pos[1] - norm_pos[0]) / delta_t, -v_clip, v_clip)
        kinematics[-2, :, 3:6] = np.clip((norm_pos[-1] - norm_pos[-3]) / (2.0 * delta_t), -v_clip, v_clip)
        kinematics[-1, :, 3:6] = np.clip((norm_pos[-1] - norm_pos[-2]) / delta_t, -v_clip, v_clip)

        a_mid = (2.0 * p0 - p1 - 2.0 * p2 - p3 + 2.0 * p4) / (7.0 * (delta_t ** 2))
        kinematics[2:-2, :, 6:9] = np.clip(a_mid, -a_clip, a_clip)
        a1 = np.clip((norm_pos[2] - 2.0 * norm_pos[1] + norm_pos[0]) / (delta_t ** 2), -a_clip, a_clip)
        kinematics[0, :, 6:9] = a1
        kinematics[1, :, 6:9] = a1
        a_end = np.clip((norm_pos[-1] - 2.0 * norm_pos[-2] + norm_pos[-3]) / (delta_t ** 2), -a_clip, a_clip)
        kinematics[-2, :, 6:9] = a_end
        kinematics[-1, :, 6:9] = a_end
    else:
        kinematics[:, :, 3:9] = 0.0

    # 3. Standardized 19D Phonology
    phon = np.zeros((T, 19), dtype=np.float32)
    lh_w, lh_idx, lh_pky = 0, 5, 17
    rh_w, rh_idx, rh_pky = 21, 26, 38

    # Palm Normals
    u_lh = norm_pos[:, lh_idx] - norm_pos[:, lh_w]
    v_lh = norm_pos[:, lh_pky] - norm_pos[:, lh_w]
    n_lh = np.cross(u_lh, v_lh)
    n_lh_norm = np.maximum(np.linalg.norm(n_lh, axis=-1, keepdims=True), 1e-5)
    phon[:, 0:3] = n_lh / n_lh_norm

    u_rh = norm_pos[:, rh_idx] - norm_pos[:, rh_w]
    v_rh = norm_pos[:, rh_pky] - norm_pos[:, rh_w]
    n_rh = np.cross(u_rh, v_rh)
    n_rh_norm = np.maximum(np.linalg.norm(n_rh, axis=-1, keepdims=True), 1e-5)
    phon[:, 3:6] = n_rh / n_rh_norm

    # Bimanual velocity synchrony
    if not is_static:
        lh_v = kinematics[:, lh_w, 3:6]
        rh_v = kinematics[:, rh_w, 3:6]
        lh_v_n = np.maximum(np.linalg.norm(lh_v, axis=-1, keepdims=True), 1e-5)
        rh_v_n = np.maximum(np.linalg.norm(rh_v, axis=-1, keepdims=True), 1e-5)
        phon[:, 6:7] = np.sum((lh_v / lh_v_n) * (rh_v / rh_v_n), axis=-1, keepdims=True)

    # Location Anchoring to Face via Gaussian Contact Kernel k_face in (0, 1]
    face_cen = np.mean(norm_pos[:, 48:60, :3], axis=1)  # [T, 3]
    d_lh_face = np.linalg.norm(norm_pos[:, lh_w, :3] - face_cen, axis=-1)
    d_rh_face = np.linalg.norm(norm_pos[:, rh_w, :3] - face_cen, axis=-1)
    sigma_face_sq = 2.0 * (0.35 ** 2)
    phon[:, 7] = np.exp(- (d_lh_face ** 2) / sigma_face_sq)
    phon[:, 8] = np.exp(- (d_rh_face ** 2) / sigma_face_sq)

    # Finger Curl bounded in [0, 1] normalized by palm length
    lh_tips = [4, 8, 12, 16, 20]
    rh_tips = [25, 29, 33, 37, 41]
    lh_palm_len = np.maximum(np.linalg.norm(norm_pos[:, 9, :3] - norm_pos[:, 0, :3], axis=-1, keepdims=True), 1e-4)
    rh_palm_len = np.maximum(np.linalg.norm(norm_pos[:, 30, :3] - norm_pos[:, 21, :3], axis=-1, keepdims=True), 1e-4)

    diff_lh = norm_pos[:, lh_tips, :3] - norm_pos[:, 0:1, :3]
    phon[:, 9:14] = np.clip(np.linalg.norm(diff_lh, axis=-1) / (1.85 * lh_palm_len), 0.0, 1.0)

    diff_rh = norm_pos[:, rh_tips, :3] - norm_pos[:, 21:22, :3]
    phon[:, 14:19] = np.clip(np.linalg.norm(diff_rh, axis=-1) / (1.85 * rh_palm_len), 0.0, 1.0)

    return kinematics, phon



# ==============================================================================
# 2. TOP-LEVEL PARALLEL WORKER FUNCTIONS
# ==============================================================================

def _worker_how2sign(task_arg: Tuple[str, str, str]) -> Optional[Dict[str, Any]]:
    """Worker task to parse a single How2Sign .npy file."""
    fp_str, text, split = task_arg
    try:
        arr = np.load(fp_str)
        if arr.ndim != 3 or arr.shape[1] != 543:
            return None
        pos60 = convert_543_to_60(arr)
        kin, phon = compute_kinematics_and_phonology(pos60, is_static=False)
        stem = Path(fp_str).stem.replace("_holistic", "")
        return {
            "id": stem,
            "features": torch.from_numpy(kin).half(),
            "phonology": torch.from_numpy(phon).half(),
            "label": text,
            "text": text,
            "task": "sentence_level",
            "source": "How2Sign",
            "split": split,
        }
    except Exception:
        return None


# Global worker model instance (initialized once per worker process)
_GLOBAL_WB = None

def _init_worker_model(mode: str = 'light', device_mode: str = 'cpu', num_gpus: int = 0):
    global _GLOBAL_WB
    if _RTMLIB_AVAILABLE and _GLOBAL_WB is None:
        try:
            try:
                import onnxruntime as ort
                ort.set_default_logger_severity(3)
            except Exception:
                pass

            if device_mode == 'cuda' and num_gpus > 0:
                worker_id = (mp.current_process()._identity[0] - 1) if (mp.current_process()._identity and len(mp.current_process()._identity) > 0) else 0
                assigned_device = f"cuda:{worker_id % num_gpus}"
            else:
                assigned_device = 'cpu'

            try:
                _GLOBAL_WB = Wholebody(to_openpose=False, mode=mode, backend='onnxruntime', device=assigned_device)
            except Exception:
                # Fallback to cpu if GPU provider initialization fails on this worker
                _GLOBAL_WB = Wholebody(to_openpose=False, mode=mode, backend='onnxruntime', device='cpu')
        except Exception:
            _GLOBAL_WB = None


def _worker_image(task_arg: Tuple[str, str, str, str, str]) -> Optional[Dict[str, Any]]:
    """Worker task to run Wholebody pose on a static image and repeat for T=40."""
    global _GLOBAL_WB
    img_p_str, label, task_type, source_name, split = task_arg
    if _GLOBAL_WB is None or cv2 is None:
        return None
    try:
        img = cv2.imread(img_p_str)
        if img is None:
            return None
        h, w = img.shape[:2]
        if w > 256:
            img = cv2.resize(img, (256, int(h * (256.0 / w))))
        kpts, _ = _GLOBAL_WB(img)
        kpts_pt = kpts[0] if (kpts is not None and len(kpts) > 0) else np.zeros((133, 2), dtype=np.float32)

        # Repeat for exactly T=40 static frames
        kpts_seq = np.repeat(kpts_pt[np.newaxis, :, :], 40, axis=0)
        pos60 = convert_133_to_60(kpts_seq)
        kin, phon = compute_kinematics_and_phonology(pos60, is_static=True)

        return {
            "id": Path(img_p_str).name,
            "features": torch.from_numpy(kin).half(),
            "phonology": torch.from_numpy(phon).half(),
            "label": label,
            "raw_label_str": label,
            "text": label,
            "task": task_type,
            "source": source_name,
            "split": split,
        }
    except Exception:
        return None


def _worker_video(task_arg: Tuple[str, str, str, str, str, str, int, int]) -> Optional[Dict[str, Any]]:
    """Worker task to decode a video, run Wholebody on sampled frames (min 40 frames)."""
    global _GLOBAL_WB
    vid_p_str, gloss, lex_code, p_id, split, source_name, f_start, f_end = task_arg
    if _GLOBAL_WB is None or cv2 is None:
        return None
    try:
        cap = cv2.VideoCapture(vid_p_str)
        all_f = []
        while True:
            ret, fr = cap.read()
            if not ret:
                break
            all_f.append(fr)
        cap.release()

        if not all_f:
            return None

        if f_end > f_start and f_end <= len(all_f):
            clip_f = all_f[f_start:f_end]
        else:
            clip_f = all_f

        # Subsample if video is overly long (keep <= 64 frames for efficiency);
        # Short clips (len < 40) are smoothly linearly interpolated in the 3D landmark domain
        # inside compute_kinematics_and_phonology, avoiding 2.5x redundant Wholebody neural inferences.
        if len(clip_f) > 64:
            idx_sub = np.linspace(0, len(clip_f) - 1, 64, dtype=int)
            clip_f = [clip_f[i] for i in idx_sub]

        kpts_list = []
        for frame in clip_f:
            h, w = frame.shape[:2]
            if w > 384:
                frame = cv2.resize(frame, (384, int(h * (384.0 / w))))
            kpts, _ = _GLOBAL_WB(frame)
            if kpts is not None and len(kpts) > 0:
                kpts_list.append(kpts[0])
            else:
                kpts_list.append(np.zeros((133, 2), dtype=np.float32))

        kpts_arr = np.array(kpts_list, dtype=np.float32)
        pos60 = convert_133_to_60(kpts_arr)
        kin, phon = compute_kinematics_and_phonology(pos60, is_static=False)

        rec = {
            "id": Path(vid_p_str).name,
            "features": torch.from_numpy(kin).half(),
            "phonology": torch.from_numpy(phon).half(),
            "label": gloss,
            "raw_label_str": gloss,
            "text": gloss,
            "task": "isolated_gloss",
            "source": source_name,
            "split": split,
        }
        if lex_code:
            rec["asl_lex"] = lex_code
        if p_id:
            rec["signer_id"] = p_id
        return rec
    except Exception:
        return None


def _worker_frame_folder(task_arg: Tuple[str, str, str, str, str]) -> Optional[Dict[str, Any]]:
    """Worker task to run Wholebody pose on a sequence folder of frame images."""
    global _GLOBAL_WB
    folder_p_str, label, task_type, source_name, split = task_arg
    if _GLOBAL_WB is None or cv2 is None:
        return None
    try:
        folder_p = Path(folder_p_str)
        img_files = sorted(list(folder_p.glob("*.jpg")) + list(folder_p.glob("*.png")))
        if not img_files:
            return None
        if len(img_files) > 64:
            idx_sub = np.linspace(0, len(img_files) - 1, 64, dtype=int)
            img_files = [img_files[i] for i in idx_sub]

        kpts_list = []
        for img_p in img_files:
            img = cv2.imread(str(img_p))
            if img is None:
                kpts_list.append(np.zeros((133, 2), dtype=np.float32))
                continue
            h, w = img.shape[:2]
            if w > 384:
                img = cv2.resize(img, (384, int(h * (384.0 / w))))
            kpts, _ = _GLOBAL_WB(img)
            if kpts is not None and len(kpts) > 0:
                kpts_list.append(kpts[0])
            else:
                kpts_list.append(np.zeros((133, 2), dtype=np.float32))

        kpts_arr = np.array(kpts_list, dtype=np.float32)
        pos60 = convert_133_to_60(kpts_arr)
        kin, phon = compute_kinematics_and_phonology(pos60, is_static=False)

        return {
            "id": folder_p.name,
            "features": torch.from_numpy(kin).half(),
            "phonology": torch.from_numpy(phon).half(),
            "label": label,
            "raw_label_str": label,
            "text": label,
            "task": task_type,
            "source": source_name,
            "split": split,
        }
    except Exception:
        return None


# ==============================================================================
# 3. STREAMING ZIP ARCHIVE SHARD WRITER
# ==============================================================================

class ZipShardWriter:
    def __init__(self, zip_path: Union[str, Path], compress_level: int = 1):
        self.zip_path = Path(zip_path)
        self.zip_path.parent.mkdir(parents=True, exist_ok=True)
        self.compress_level = compress_level
        self.zf = None
        self._open()
        self.written_shards = set(self.zf.namelist())
        self.total_samples_written = 0
        self.total_bytes_uncompressed = 0

    def _open(self) -> None:
        if self.zf is None:
            self.zf = zipfile.ZipFile(
                self.zip_path,
                mode='a' if self.zip_path.exists() else 'w',
                compression=zipfile.ZIP_DEFLATED,
                compresslevel=self.compress_level
            )

    def reopen(self) -> None:
        """Reopens the zip archive in append mode after a checkpoint close."""
        if self.zf is None:
            self._open()

    def get_existing_shard_counts(self) -> Dict[str, int]:
        """Finds highest shard index for each split already in zip."""
        counts = {"train": 0, "val": 0, "test": 0}
        for name in self.written_shards:
            for sp in ["train", "val", "test"]:
                if name.startswith(f"{sp}/shard_") and name.endswith(".pt"):
                    try:
                        idx_str = name.split("shard_")[-1].replace(".pt", "")
                        idx = int(idx_str)
                        if idx >= counts[sp]:
                            counts[sp] = idx + 1
                    except ValueError:
                        pass
        return counts

    def get_existing_manifest(self) -> Dict[str, Any]:
        """Reads existing manifest if present in zip."""
        if "manifest.json" in self.written_shards:
            try:
                return json.loads(self.zf.read("manifest.json").decode("utf-8"))
            except Exception:
                pass
        return {}

    def add_shard(self, split: str, shard_idx: int, records: List[Dict[str, Any]]) -> str:
        if self.zf is None:
            self._open()
        arcname = f"{split}/shard_{shard_idx:04d}.pt"
        buf = io.BytesIO()
        torch.save(records, buf)
        raw_bytes = buf.getvalue()

        self.zf.writestr(arcname, raw_bytes)
        self.written_shards.add(arcname)
        self.total_samples_written += len(records)
        self.total_bytes_uncompressed += len(raw_bytes)

        self.zf.fp.flush()
        del buf, raw_bytes
        return arcname

    def add_json(self, arcname: str, data: Any) -> None:
        if self.zf is None:
            self._open()
        content = json.dumps(data, indent=2, ensure_ascii=False).encode('utf-8')
        self.zf.writestr(arcname, content)
        self.zf.fp.flush()

    def get_current_zip_size_mb(self) -> float:
        if self.zip_path.exists():
            return self.zip_path.stat().st_size / (1024 * 1024)
        return 0.0

    def close(self) -> None:
        if self.zf is not None:
            self.zf.close()
            self.zf = None


# ==============================================================================
# 4. HIGH-THROUGHPUT ORCHESTRATOR
# ==============================================================================

def main():
    parser = argparse.ArgumentParser(description="High-Throughput ASL Preprocessor (All 96 vCPUs, No Capping, Min 40 Frames)")
    parser.add_argument("--output-zip", type=str, default="/kaggle/working/asl_unified_shards.zip")
    parser.add_argument("--shard-size", type=int, default=2000)
    parser.add_argument("--workers", type=int, default=0, help="Number of worker processes (0=auto-detect: 4 per GPU for CUDA, 48/host CPUs for CPU)")
    parser.add_argument("--device", type=str, default="auto", choices=["auto", "cuda", "cpu"], help="Compute device ('auto' auto-detects CUDA GPUs; 'cuda' forces GPU; 'cpu' runs CPU workers)")
    parser.add_argument("--mode", type=str, default="light", choices=["light", "balanced"], help="Wholebody detector/pose mode ('light' runs 8-10x faster)")
    parser.add_argument("--use-shm", action="store_true", default=True, help="Build the zip archive in /dev/shm (RAM) for ultra-fast memory I/O, then sync to target")
    parser.add_argument("--shm-dir", type=str, default="/dev/shm", help="Path to shared memory / tmpfs directory")
    parser.add_argument("--datasets", nargs="+", default=["alphabet", "numbers", "citizen", "wlasl", "chicagofswild"], help="List of datasets to preprocess (how2sign excluded by default)")
    parser.add_argument("--disable-how2sign", action="store_true", default=True, help="Disable How2Sign preprocessing (default: True; downloading video sequences separately)")
    parser.add_argument("--enable-how2sign", dest="disable_how2sign", action="store_false", help="Explicitly enable How2Sign preprocessing")
    parser.add_argument("--compress-level", type=int, default=1, help="Deflate compression level (1=fastest, 6=standard, 9=maximum)")
    parser.add_argument("--log-file", type=str, default="/kaggle/working/preprocess.log")
    args = parser.parse_args()

    log_path = Path(args.log_file)
    log_path.parent.mkdir(parents=True, exist_ok=True)

    def log_msg(msg: str):
        t_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        formatted = f"[{t_str}] {msg}"
        print(formatted, flush=True)
        try:
            with open(log_path, "a", encoding="utf-8") as lf:
                lf.write(formatted + "\n")
        except Exception:
            pass

    # Hardware detection & device resolution (CUDA GPU vs CPU)
    device_mode = "cpu"
    num_gpus = 0
    if args.device in ("auto", "cuda") and torch.cuda.is_available():
        try:
            import onnxruntime as ort
            if "CUDAExecutionProvider" in ort.get_available_providers():
                device_mode = "cuda"
                num_gpus = torch.cuda.device_count()
            else:
                if args.device == "cuda":
                    log_msg("[GPU Warning] CUDA GPU detected in PyTorch, but 'CUDAExecutionProvider' is NOT installed in onnxruntime! Falling back to CPU. (Run: pip install onnxruntime-gpu)")
        except Exception:
            pass

    # Worker allocation:
    # On GPU: 4 workers per GPU perfectly saturates TensorRT/CUDA without VRAM fragmentation (e.g. 8 workers for 2x T4).
    # On CPU: up to 48 workers or host CPU count.
    if device_mode == "cuda":
        gpu_names = [torch.cuda.get_device_name(i) for i in range(num_gpus)]
        if args.workers > 0:
            num_workers = args.workers
        else:
            num_workers = max(2, num_gpus * 4)
        worker_info = f"CUDA GPU Acceleration ({num_gpus} GPU(s): {gpu_names}, {num_workers} parallel workers across devices)"
    else:
        if args.workers > 0:
            num_workers = min(args.workers, os.cpu_count() or 4)
        else:
            num_workers = min(48, os.cpu_count() or 4)
        worker_info = f"CPU Mode ({num_workers} parallel workers, Host CPUs: {os.cpu_count()})"

    target_zip = Path(args.output_zip)
    use_shm = args.use_shm and Path(args.shm_dir).exists() and os.name != "nt"
    if use_shm:
        # Check available space in shm_dir (needs >= 20 GB to safely store large shard archives)
        try:
            import shutil
            _, _, shm_free = shutil.disk_usage(args.shm_dir)
            if shm_free < 20 * 1024 * 1024 * 1024:
                log_msg(f"[RAM Mode Notice] {args.shm_dir} has only {shm_free / (1024**3):.1f} GB free (< 20 GB). Writing directly to target disk {target_zip} to prevent memory exhaustion.")
                use_shm = False
        except Exception:
            pass

    if use_shm:
        build_zip = Path(args.shm_dir) / target_zip.name
        log_msg(f"[RAM Mode] Accelerating I/O via {build_zip} (RAM tmpfs / 330GB RAM)!")
        if target_zip.exists() and not build_zip.exists():
            log_msg(f"[RAM Mode] Copying existing {target_zip} into RAM ({target_zip.stat().st_size / (1024*1024):.1f} MB)...")
            import shutil
            shutil.copy2(target_zip, build_zip)
    else:
        build_zip = target_zip

    log_msg("=================================================================")
    log_msg("   HIGH-THROUGHPUT ASL PREPROCESSOR & ZIP STREAMING ENGINE       ")
    log_msg(f"  Execution Engine    : {worker_info}")
    log_msg(f"  Wholebody Mode      : {args.mode}")
    log_msg(f"  RAM Storage Accel   : {'ENABLED (' + str(build_zip) + ')' if use_shm else 'DISABLED (Disk direct)'}")
    log_msg(f"  Policy: NO CAPPING (All Samples across all classes)")
    log_msg(f"  Policy: MINIMUM 40 FRAMES (Static Hand T=40, Video T>=40)")
    log_msg(f"  Target Zip Archive  : {args.output_zip}")
    log_msg(f"  Shard Size          : {args.shard_size} samples")
    log_msg("=================================================================")

    writer = ZipShardWriter(build_zip, compress_level=args.compress_level)
    t_start = time.time()
    
    shard_counters = writer.get_existing_shard_counts()
    existing_manifest = writer.get_existing_manifest()
    total_stats = dict(existing_manifest.get("dataset_breakdown", {}))
    writer.total_samples_written = existing_manifest.get("total_samples", 0)
    writer.total_bytes_uncompressed = existing_manifest.get("uncompressed_bytes", 0)

    if shard_counters["train"] > 0 or shard_counters["val"] > 0:
        log_msg(f"[Resume] Detected existing shards in archive: train={shard_counters['train']}, val={shard_counters['val']}, test={shard_counters['test']}")
        if total_stats:
            log_msg(f"[Resume] Existing dataset breakdown: {total_stats}")

    buffers = {"train": [], "val": [], "test": []}

    def _sync_to_disk():
        if use_shm and build_zip.exists():
            import shutil
            log_msg(f"[RAM Sync] Checkpoint sync ({build_zip.stat().st_size / (1024*1024):.1f} MB) from RAM to {target_zip}...")
            writer.close()
            target_zip.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(build_zip, target_zip)
            writer.reopen()

    # Helper to flush buffer to zip
    def _check_flush(split, force=False):
        if len(buffers[split]) >= args.shard_size or (force and buffers[split]):
            arc = writer.add_shard(split, shard_counters[split], buffers[split])
            log_msg(f"[*] Wrote {arc} ({len(buffers[split])} samples). Total samples: {writer.total_samples_written}, Zip: {writer.get_current_zip_size_mb():.1f} MB")
            shard_counters[split] += 1
            buffers[split] = []

    # --------------------------------------------------------------------------
    # 1. HOW2SIGN (35k sequences) - Parallel .npy ingestion
    # --------------------------------------------------------------------------
    if "how2sign" in args.datasets and not args.disable_how2sign:
        if total_stats.get("How2Sign", 0) >= 30000 or (shard_counters["train"] >= 16 and "How2Sign" in total_stats):
            log_msg(f"\n>>> [1/6] HOW2SIGN: ALREADY COMPLETED in archive ({total_stats.get('How2Sign')} samples). Skipping re-extraction.")
        else:
            h2s_root = Path("/kaggle/input/datasets/psewmuthu/how2sign-holistic")
            if not h2s_root.exists():
                h2s_root = Path("/kaggle/input/how2sign-holistic")
            feat_root = h2s_root / "how2sign_holistic_features" if (h2s_root / "how2sign_holistic_features").exists() else h2s_root
            if feat_root.exists():
                log_msg("\n>>> [1/6] PREPROCESSING HOW2SIGN (35,000 continuous sentences)...")
                text_map = {}
                for f in (feat_root / "metadata").glob("*.csv"):
                    try:
                        df = pd.read_csv(f, sep='\t')
                        for _, row in df.iterrows():
                            name = str(row.get('SENTENCE_NAME', '')).strip()
                            sent = str(row.get('SENTENCE', '')).strip()
                            if name and sent:
                                text_map[name] = sent
                                text_map[name.replace("-rgb_front", "").replace("_rgb_front", "")] = sent
                    except Exception:
                        pass

                h2s_tasks = []
                for split in ["train", "val", "test"]:
                    sp_dir = feat_root / split / "frontal"
                    if sp_dir.exists():
                        for fp in sp_dir.glob("*.npy"):
                            stem = fp.stem.replace("_holistic", "")
                            text = text_map.get(stem, text_map.get(stem.replace("-rgb_front", ""), "how2sign_continuous"))
                            h2s_tasks.append((str(fp), text, split))

                log_msg(f"[How2Sign] Prepared {len(h2s_tasks)} tasks. Launching pool with {num_workers} workers...")
                t0 = time.time()
                h2s_count = 0
                with mp.Pool(processes=num_workers) as pool:
                    for res in pool.imap_unordered(_worker_how2sign, h2s_tasks, chunksize=128):
                        if res is not None:
                            sp = res["split"]
                            buffers[sp].append(res)
                            h2s_count += 1
                            _check_flush(sp)
                            if h2s_count % 5000 == 0:
                                dt_h = time.time() - t0
                                log_msg(f"[How2Sign] Progress: {h2s_count}/{len(h2s_tasks)} ({h2s_count/dt_h:.1f} samples/sec)")

                for sp in ["train", "val", "test"]:
                    _check_flush(sp, force=True)
                total_stats["How2Sign"] = h2s_count
                log_msg(f"[How2Sign] COMPLETE! Ingested {h2s_count} samples in {time.time()-t0:.1f}s.")
                _sync_to_disk()
    else:
        log_msg("\n>>> [1/6] HOW2SIGN: SKIPPED (Disabled by configuration; downloading video sequences separately).")

    # --------------------------------------------------------------------------
    # 2. ASL ALPHABET (87,000 images) - Parallel Wholebody Pose (T=40 static)
    # --------------------------------------------------------------------------
    if "alphabet" in args.datasets:
        if total_stats.get("ASL_Alphabet", 0) >= 80000:
            log_msg(f"\n>>> [2/6] ASL ALPHABET: ALREADY COMPLETED in archive ({total_stats.get('ASL_Alphabet')} samples). Skipping.")
        else:
            alpha_root = Path("/kaggle/input/datasets/grassknoted/asl-alphabet")
            if not alpha_root.exists():
                alpha_root = Path("/kaggle/input/asl-alphabet")
            train_dir = alpha_root / "asl_alphabet_train" / "asl_alphabet_train" if (alpha_root / "asl_alphabet_train" / "asl_alphabet_train").exists() else alpha_root / "asl_alphabet_train"
            if train_dir.exists() and _RTMLIB_AVAILABLE:
                log_msg(f"\n>>> [2/6] PREPROCESSING ASL ALPHABET (87,000 images, T=40 static hand, mode={args.mode})...")
                classes = sorted([d for d in train_dir.iterdir() if d.is_dir()])
                alpha_tasks = []
                for cls_dir in classes:
                    label = cls_dir.name
                    imgs = sorted(list(cls_dir.glob("*.jpg")) + list(cls_dir.glob("*.png")))
                    for img_p in imgs:
                        alpha_tasks.append((str(img_p), label, "fingerspelling_char", "ASL_Alphabet", "train"))

                log_msg(f"[Alphabet] Found {len(alpha_tasks)} total images across {len(classes)} classes. Launching {num_workers} workers...")
                t0 = time.time()
                alpha_count = 0
                with mp.Pool(processes=num_workers, initializer=_init_worker_model, initargs=(args.mode, device_mode, num_gpus)) as pool:
                    for res in pool.imap_unordered(_worker_image, alpha_tasks, chunksize=64):
                        if res is not None:
                            buffers["train"].append(res)
                            alpha_count += 1
                            _check_flush("train")
                            if alpha_count % 5000 == 0:
                                dt_a = time.time() - t0
                                log_msg(f"[Alphabet] Progress: {alpha_count}/{len(alpha_tasks)} ({alpha_count/dt_a:.1f} samples/sec)")

                _check_flush("train", force=True)
                total_stats["ASL_Alphabet"] = alpha_count
                log_msg(f"[Alphabet] COMPLETE! Ingested {alpha_count} samples in {time.time()-t0:.1f}s.")
                _sync_to_disk()

    # --------------------------------------------------------------------------
    # 3. SYNTHETIC ASL NUMBERS (~10,000 images) - Parallel Wholebody (T=40 static)
    # --------------------------------------------------------------------------
    if "numbers" in args.datasets:
        if total_stats.get("Synthetic_Numbers", 0) >= 9000:
            log_msg(f"\n>>> [3/6] SYNTHETIC NUMBERS: ALREADY COMPLETED in archive ({total_stats.get('Synthetic_Numbers')} samples). Skipping.")
        else:
            num_root = Path("/kaggle/input/datasets/lexset/synthetic-asl-numbers")
            if not num_root.exists():
                num_root = Path("/kaggle/input/synthetic-asl-numbers")
            if num_root.exists() and _RTMLIB_AVAILABLE:
                log_msg(f"\n>>> [3/6] PREPROCESSING SYNTHETIC NUMBERS (T=40 static hand, mode={args.mode})...")
                num_tasks = []
                for split_name, sub in [("train", "Train_Nums"), ("test", "Test_Nums")]:
                    sub_dir = num_root / sub
                    if not sub_dir.exists():
                        continue
                    for digit_dir in sorted([d for d in sub_dir.iterdir() if d.is_dir()]):
                        digit = digit_dir.name
                        for img_p in sorted(list(digit_dir.glob("*.jpg")) + list(digit_dir.glob("*.png"))):
                            num_tasks.append((str(img_p), digit, "fingerspelling_digit", "Synthetic_Numbers", split_name))

                log_msg(f"[Numbers] Found {len(num_tasks)} images. Launching {num_workers} workers...")
                t0 = time.time()
                num_count = 0
                with mp.Pool(processes=num_workers, initializer=_init_worker_model, initargs=(args.mode, device_mode, num_gpus)) as pool:
                    for res in pool.imap_unordered(_worker_image, num_tasks, chunksize=64):
                        if res is not None:
                            sp = res["split"]
                            buffers[sp].append(res)
                            num_count += 1
                            _check_flush(sp)

                for sp in ["train", "test"]:
                    _check_flush(sp, force=True)
                total_stats["Synthetic_Numbers"] = num_count
                log_msg(f"[Numbers] COMPLETE! Ingested {num_count} samples in {time.time()-t0:.1f}s.")
                _sync_to_disk()

    # --------------------------------------------------------------------------
    # 4. WLASL PROCESSED (11,980 videos) - Parallel Video Wholebody (T >= 40)
    # --------------------------------------------------------------------------
    if "wlasl" in args.datasets:
        if total_stats.get("WLASL", 0) >= 10000:
            log_msg(f"\n>>> [4/6] WLASL: ALREADY COMPLETED in archive ({total_stats.get('WLASL')} samples). Skipping.")
        else:
            wlasl_root = Path("/kaggle/input/datasets/risangbaskoro/wlasl-processed")
            if not wlasl_root.exists():
                wlasl_root = Path("/kaggle/input/wlasl-processed")
            json_p = wlasl_root / "WLASL_v0.3.json"
            vid_dir = wlasl_root / "videos"
            if json_p.exists() and vid_dir.exists() and _RTMLIB_AVAILABLE:
                log_msg(f"\n>>> [4/6] PREPROCESSING WLASL (11,980 videos, T >= 40, mode={args.mode})...")
                with open(json_p, "r", encoding="utf-8") as f:
                    wdata = json.load(f)

                wlasl_tasks = []
                for entry in wdata:
                    gloss = entry.get("gloss", "")
                    for inst in entry.get("instances", []):
                        vid_id = inst.get("video_id", "")
                        split = inst.get("split", "train")
                        vid_p = vid_dir / f"{vid_id}.mp4"
                        if vid_p.exists():
                            f_start = inst.get("frame_start", 1) - 1
                            f_end = inst.get("frame_end", -1)
                            wlasl_tasks.append((str(vid_p), gloss, "", "", split, "WLASL", f_start, f_end))

                log_msg(f"[WLASL] Prepared {len(wlasl_tasks)} video tasks. Launching {num_workers} workers...")
                t0 = time.time()
                w_count = 0
                with mp.Pool(processes=num_workers, initializer=_init_worker_model, initargs=(args.mode, device_mode, num_gpus)) as pool:
                    for res in pool.imap_unordered(_worker_video, wlasl_tasks, chunksize=8):
                        if res is not None:
                            sp = res["split"]
                            buffers[sp].append(res)
                            w_count += 1
                            _check_flush(sp)
                            if w_count % 1000 == 0:
                                dt_w = time.time() - t0
                                log_msg(f"[WLASL] Progress: {w_count}/{len(wlasl_tasks)} ({w_count/dt_w:.1f} vids/sec)")

                for sp in ["train", "val", "test"]:
                    _check_flush(sp, force=True)
                total_stats["WLASL"] = w_count
                log_msg(f"[WLASL] COMPLETE! Ingested {w_count} videos in {time.time()-t0:.1f}s.")
                _sync_to_disk()

    # --------------------------------------------------------------------------
    # 5. ASL CITIZEN (83,399 videos) - Parallel Video Wholebody (T >= 40)
    # --------------------------------------------------------------------------
    if "citizen" in args.datasets:
        if total_stats.get("ASL_Citizen", 0) >= 70000:
            log_msg(f"\n>>> [5/6] ASL CITIZEN: ALREADY COMPLETED in archive ({total_stats.get('ASL_Citizen')} samples). Skipping.")
        else:
            cit_root = Path("/kaggle/input/datasets/abd0kamel/asl-citizen")
            if not cit_root.exists():
                cit_root = Path("/kaggle/input/asl-citizen")
            cit_base = cit_root / "ASL_Citizen" if (cit_root / "ASL_Citizen").exists() else cit_root
            vid_dir = cit_base / "videos"
            splits_dir = cit_base / "splits"
            if vid_dir.exists() and splits_dir.exists() and _RTMLIB_AVAILABLE:
                log_msg(f"\n>>> [5/6] PREPROCESSING ASL CITIZEN (83,399 videos, T >= 40, mode={args.mode})...")
                cit_tasks = []
                for split in ["train", "val", "test"]:
                    csv_p = splits_dir / f"{split}.csv"
                    if csv_p.exists():
                        df = pd.read_csv(csv_p)
                        for _, row in df.iterrows():
                            v_file = str(row['Video file']).strip()
                            gloss = str(row['Gloss']).strip()
                            lex = str(row.get('ASL-LEX Code', '')).strip()
                            pid = str(row.get('Participant ID', '')).strip()
                            vp = vid_dir / v_file
                            if vp.exists():
                                cit_tasks.append((str(vp), gloss, lex, pid, split, "ASL_Citizen", 0, -1))

                log_msg(f"[ASL Citizen] Prepared {len(cit_tasks)} video tasks. Launching {num_workers} workers...")
                t0 = time.time()
                c_count = 0
                with mp.Pool(processes=num_workers, initializer=_init_worker_model, initargs=(args.mode, device_mode, num_gpus)) as pool:
                    for res in pool.imap_unordered(_worker_video, cit_tasks, chunksize=8):
                        if res is not None:
                            sp = res["split"]
                            buffers[sp].append(res)
                            c_count += 1
                            _check_flush(sp)
                            if c_count % 2000 == 0:
                                dt_c = time.time() - t0
                                log_msg(f"[ASL Citizen] Progress: {c_count}/{len(cit_tasks)} ({c_count/dt_c:.1f} vids/sec)")

                for sp in ["train", "val", "test"]:
                    _check_flush(sp, force=True)
                total_stats["ASL_Citizen"] = c_count
                log_msg(f"[ASL Citizen] COMPLETE! Ingested {c_count} videos in {time.time()-t0:.1f}s.")
                _sync_to_disk()

    # --------------------------------------------------------------------------
    # 6. CHICAGOFSWILD (Fingerspelling Sequences)
    # --------------------------------------------------------------------------
    if "chicagofswild" in args.datasets or "chicago" in args.datasets:
        if total_stats.get("ChicagoFSWild", 0) >= 5000:
            log_msg(f"\n>>> [6/6] CHICAGOFSWILD: ALREADY COMPLETED in archive ({total_stats.get('ChicagoFSWild')} samples). Skipping.")
        else:
            log_msg("\n>>> [6/6] PREPROCESSING CHICAGOFSWILD (Fingerspelling)...")
            pre_extracted_candidates = [
                Path("/kaggle/input/datasets/tranquocbao2012/frakenstein-asl-final-version/asl_dataset/asl_preprocessed_phase1"),
                Path("/kaggle/input/frakenstein-asl-final-version/asl_dataset/asl_preprocessed_phase1"),
                Path("/kaggle/input/datasets/tranquocbao2012/frakenstein-asl/asl_preprocessed_phase1"),
                Path("/kaggle/input/frakenstein-asl/asl_preprocessed_phase1"),
            ]
            found_pre = None
            for cand in pre_extracted_candidates:
                if cand.exists() and (cand / "train").exists():
                    found_pre = cand
                    break

            if found_pre is not None:
                log_msg(f"[ChicagoFSWild] Found pre-extracted Frankenstein shards at {found_pre}! Streaming records directly...")
                chicago_count = 0
                for sp in ["train", "val", "test"]:
                    sp_dir = found_pre / sp
                    if sp_dir.exists():
                        for pt_file in sorted(sp_dir.glob("*.pt")):
                            try:
                                shard_data = torch.load(pt_file, map_location="cpu", weights_only=False)
                                recs = list(shard_data.values()) if isinstance(shard_data, dict) else shard_data
                                for r in recs:
                                    if isinstance(r, dict):
                                        src = str(r.get("source", "")).lower()
                                        if "chicago" in src or "cfs" in src:
                                            buffers[sp].append(r)
                                            chicago_count += 1
                                            _check_flush(sp)
                            except Exception:
                                pass
                    _check_flush(sp, force=True)
                total_stats["ChicagoFSWild"] = chicago_count
                log_msg(f"[ChicagoFSWild] COMPLETE! Ingested {chicago_count} pre-extracted samples.")
                _sync_to_disk()
            else:
                cf_root = Path("/kaggle/input/datasets/joebeachcapital/chicagofswild")
                if not cf_root.exists():
                    cf_root = Path("/kaggle/input/chicagofswild")
                csv_p = cf_root / "ChicagoFSWild.csv"
                frames_dir = cf_root / "ChicagoFSWild-Frames"
                if not frames_dir.exists() and (cf_root / "ChicagoFSWild-Frames" / "ChicagoFSWild-Frames").exists():
                    frames_dir = cf_root / "ChicagoFSWild-Frames" / "ChicagoFSWild-Frames"

                if csv_p.exists() and frames_dir.exists() and _RTMLIB_AVAILABLE:
                    log_msg(f"[ChicagoFSWild] Found CSV metadata and frames at {frames_dir}. Launching {num_workers} workers...")
                    df = pd.read_csv(csv_p)
                    cfs_tasks = []
                    for _, row in df.iterrows():
                        folder_name = str(row.get('filename', row.get('url', ''))).strip()
                        label = str(row.get('label', '')).strip()
                        part = str(row.get('partition', 'train')).strip().lower()
                        split = "val" if "dev" in part or "val" in part else ("test" if "test" in part else "train")
                        seq_folder = frames_dir / folder_name
                        if seq_folder.exists() and label:
                            cfs_tasks.append((str(seq_folder), label, "fingerspelling_seq", "ChicagoFSWild", split))

                    if cfs_tasks:
                        log_msg(f"[ChicagoFSWild] Prepared {len(cfs_tasks)} sequence tasks. Launching pool...")
                        t0 = time.time()
                        cfs_count = 0
                        with mp.Pool(processes=num_workers, initializer=_init_worker_model, initargs=(args.mode, device_mode, num_gpus)) as pool:
                            for res in pool.imap_unordered(_worker_frame_folder, cfs_tasks, chunksize=8):
                                if res is not None:
                                    sp = res["split"]
                                    buffers[sp].append(res)
                                    cfs_count += 1
                                    _check_flush(sp)
                                    if cfs_count % 1000 == 0:
                                        dt_cfs = time.time() - t0
                                        log_msg(f"[ChicagoFSWild] Progress: {cfs_count}/{len(cfs_tasks)} ({cfs_count/dt_cfs:.1f} seq/s)")

                        for sp in ["train", "val", "test"]:
                            _check_flush(sp, force=True)
                        total_stats["ChicagoFSWild"] = cfs_count
                        log_msg(f"[ChicagoFSWild] COMPLETE! Ingested {cfs_count} sequence samples in {time.time()-t0:.1f}s.")
                        _sync_to_disk()
                elif csv_p.exists():
                    log_msg(f"[ChicagoFSWild] Found CSV metadata at {csv_p} (tarball mode or unextracted).")

    # --------------------------------------------------------------------------
    # WRITE MASTER MANIFEST & VOCAB
    # --------------------------------------------------------------------------
    log_msg("\n[Manifest] Writing final master manifest.json...")
    manifest = {
        "creation_time": datetime.datetime.now().isoformat(),
        "total_samples": writer.total_samples_written,
        "total_shards": len(writer.written_shards),
        "uncompressed_bytes": writer.total_bytes_uncompressed,
        "compressed_zip_bytes": build_zip.stat().st_size if build_zip.exists() else 0,
        "dataset_breakdown": total_stats,
        "shards": sorted(list(writer.written_shards)),
        "invariants": {
            "min_frames": 40,
            "static_frames": 40,
            "num_keypoints": 60,
            "channels": 9,
            "phonology_dims": 19,
            "no_capping": True
        }
    }
    writer.add_json("manifest.json", manifest)
    writer.close()

    if use_shm and build_zip.exists():
        import shutil
        log_msg(f"[RAM Final] Moving complete archive ({build_zip.stat().st_size / (1024*1024):.1f} MB) from RAM ({build_zip}) to {target_zip}...")
        target_zip.parent.mkdir(parents=True, exist_ok=True)
        if target_zip.exists():
            target_zip.unlink()
        shutil.move(str(build_zip), str(target_zip))

    total_time = time.time() - t_start
    final_sz = target_zip.stat().st_size / (1024 * 1024) if target_zip.exists() else 0

    log_msg("=================================================================")
    log_msg("       PREPROCESSING & ZIP STREAMING FINISHED SUCCESSFULLY!      ")
    log_msg(f"  Total Samples : {writer.total_samples_written}")
    log_msg(f"  Total Shards  : {len(writer.written_shards)}")
    log_msg(f"  Final Zip Size: {final_sz:.2f} MB")
    log_msg(f"  Total Time    : {total_time:.1f}s ({total_time/60:.1f}m)")
    log_msg("=================================================================")


if __name__ == "__main__":
    main()


## ⚡ Bước 5: Khởi Chạy Quá Trình Tiền Xử Lý (GPU Multi-Worker)
Lệnh này khởi động tiền xử lý:
- `--device auto`: Tự động kích hoạt GPU T4 (hoặc CPU nếu không có GPU).
- `--mode light`: Sử dụng mô hình Wholebody nhẹ (tốc độ cao hơn 8-10 lần).
- `--disable-how2sign`: Bỏ qua How2Sign như yêu cầu.
- Lưu trực tiếp thành file nén `/kaggle/working/asl_unified_shards.zip`.

In [ ]:
# Khởi chạy tiến trình tiền xử lý đa luồng với tăng tốc GPU
!python preprocessing/unified_multidataset_preprocessor.py \
    --device auto \
    --mode light \
    --disable-how2sign \
    --output-zip /kaggle/working/asl_unified_shards.zip \
    --shard-size 2000

## ✅ Bước 6: Kiểm Tra File Shards Kết Quả & Thống Kê
Đọc manifest và xác nhận tính toàn vẹn của tệp zip sau khi xử lý xong.

In [ ]:
import json
import zipfile
from pathlib import Path

zip_path = Path("/kaggle/working/asl_unified_shards.zip")

if zip_path.exists():
    file_size_mb = zip_path.stat().st_size / (1024 * 1024)
    print("=" * 65)
    print(f"  TIỀN XỬ LÝ HOÀN TẤT THÀNH CÔNG!")
    print(f"  - Vị trí tệp zip : {zip_path}")
    print(f"  - Dung lượng zip : {file_size_mb:.2f} MB ({file_size_mb/1024:.2f} GB)")
    print("=" * 65)
    
    with zipfile.ZipFile(zip_path, 'r') as zf:
        members = zf.namelist()
        shards = [m for m in members if m.endswith('.pt')]
        print(f"  - Tổng số file shard .pt : {len(shards)}")
        
        if 'manifest.json' in members:
            manifest = json.loads(zf.read('manifest.json').decode('utf-8'))
            print(f"  - Tổng số mẫu (samples)  : {manifest.get('total_samples', 0):,}")
            print("\n  Chi tiết từng bộ dữ liệu:")
            for ds_name, count in manifest.get('dataset_breakdown', {}).items():
                print(f"     * {ds_name:20s}: {count:,} mẫu")
    print("=" * 65)
else:
    print(f"[LỖI] Không tìm thấy file kết quả tại {zip_path}!")